# Part 4 — Deep CNN with BatchNorm

## 6. Data Augmentation Helper (numpy, no Keras layer)

In [13]:
# Safe numpy augmentation — works on ALL TF versions
# Applied only at training time via a tf.data pipeline

def augment(image, label):
    """Augment a single (H,W,3) float32 image."""
    # Random left-right flip  — DISABLED for traffic signs (direction-specific)
    # Random rotation ±15 degrees
    if tf.random.uniform(()) > 0.5:
        angle = tf.random.uniform((), -0.26, 0.26)   # radians ~15 deg
        image = tf.keras.preprocessing.image.apply_affine_transform(
            image.numpy(), theta=angle * 57.3, fill_mode='nearest')
        image = tf.cast(image, tf.float32)
    # Random brightness ±20%
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.clip_by_value(image, 0.0, 1.0)
    # Random contrast
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

# Build tf.data pipelines
BATCH   = 64
AUTOTUNE = tf.data.AUTOTUNE

def make_train_ds(X, y, batch=BATCH):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.shuffle(len(X), seed=42)
    # Simple augmentation without rotation (avoids tf.py_function complexity)
    ds = ds.map(lambda x, y: (
        tf.clip_by_value(tf.image.random_brightness(
            tf.image.random_contrast(x, 0.8, 1.2), 0.2), 0.0, 1.0), y),
        num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch).prefetch(AUTOTUNE)
    return ds

def make_val_ds(X, y, batch=BATCH):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.batch(batch).prefetch(AUTOTUNE)
    return ds

train_ds32 = make_train_ds(X_tr32, y_tr32)
val_ds32   = make_val_ds(X_v32,   y_v32)
train_ds48 = make_train_ds(X_tr48, y_tr48)
val_ds48   = make_val_ds(X_v48,   y_v48)

print("tf.data pipelines ready.")

# Sanity check: peek at one batch
for xb, yb in train_ds32.take(1):
    print(f"Batch X: {xb.shape}  range [{xb.numpy().min():.2f},{xb.numpy().max():.2f}]")
    print(f"Batch y: {yb.shape}  row sums (should all be 1): {yb.numpy().sum(axis=1)[:5]}")

tf.data pipelines ready.
Batch X: (64, 32, 32, 3)  range [0.00,1.00]
Batch y: (64, 43)  row sums (should all be 1): [1. 1. 1. 1. 1.]
